In [2]:
import pandas as pd

# loading csv and previewing
df = pd.read_csv("trade-register.csv", header=0)
df.head(5)

,Recipient,Supplier,Year of order,,Number ordered,.1,Weapon designation,Weapon description,Number delivered,.2,Year(s) of delivery,status,Comments,SIPRI TIV per unit,SIPRI TIV for total order,SIPRI TIV of delivered weapons
0,Afghanistan,Russia,2010,?,6.0,?,Mi-17,transport helicopter,6,NaN,2011,Second hand,Probably second-hand; Mi-17V1 version; bought ...,2.90,17.4,17.4
1,Afghanistan,Russia,2004,?,4.0,NaN,Mi-17,transport helicopter,4,?,2005,Second hand,Second-hand; aid; Mi-8MTV version,2.90,11.6,11.6
2,Afghanistan,Soviet Union,1960,?,50.0,?,BM-13 132mm,multiple rocket launcher,50,?,1960; 1961,Second hand,Second-hand,0.08,4.0,4.0
3,Afghanistan,Soviet Union,1978,?,500.0,?,T-55,tank,500,?,1979; 1980; 1981; 1982; 1983; 1984; 1985; 1986...,Second hand,Second-hand; aid,0.50,250.0,250.0
4,Afghanistan,Soviet Union,1987,?,40.0,?,MiG-21MF,fighter aircraft,40,?,1987; 1988; 1989; 1990,Second hand,Second-hand; probably incl some MiG-21UM,4.28,171.2,171.2


In [3]:
# shape
print(df.shape)

(29916, 16)


In [4]:
# wat am i working with
print(df.dtypes)
print(df.isnull().sum())
print(df['Supplier'].unique()[:30])
print(df['Recipient'].unique()[:30])
print(df['Year of order'].unique())

Recipient                             str
Supplier                              str
Year of order                       int64
                                      str
Number ordered                    float64
 .1                                   str
Weapon designation                    str
Weapon description                    str
Number delivered                    int64
 .2                                   str
Year(s) of delivery                   str
status                                str
Comments                              str
SIPRI TIV per unit                float64
SIPRI TIV for total order         float64
SIPRI TIV of delivered weapons    float64
dtype: object
Recipient                             0
Supplier                              0
Year of order                         0
                                  14180
Number ordered                      318
 .1                               17478
Weapon designation                    0
Weapon description                

In [5]:
print(sorted(df['Year of order'].unique()))

[np.int64(1940), np.int64(1943), np.int64(1945), np.int64(1946), np.int64(1947), np.int64(1948), np.int64(1949), np.int64(1950), np.int64(1951), np.int64(1952), np.int64(1953), np.int64(1954), np.int64(1955), np.int64(1956), np.int64(1957), np.int64(1958), np.int64(1959), np.int64(1960), np.int64(1961), np.int64(1962), np.int64(1963), np.int64(1964), np.int64(1965), np.int64(1966), np.int64(1967), np.int64(1968), np.int64(1969), np.int64(1970), np.int64(1971), np.int64(1972), np.int64(1973), np.int64(1974), np.int64(1975), np.int64(1976), np.int64(1977), np.int64(1978), np.int64(1979), np.int64(1980), np.int64(1981), np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int6

so we have data from 1940-2025, recipient/supplier, weapon info, # ordered, status, TIV

In [6]:
import json
from collections import defaultdict

df = pd.read_csv('trade-register.csv', header=0)
# Rename for convenience
df = df.rename(columns={
    'Year of order': 'year_order',
    'Year(s) of delivery': 'year_delivery',
    'Number ordered': 'num_ordered',
    'Number delivered': 'num_delivered',
    'Weapon designation': 'weapon_designation',
    'Weapon description': 'weapon_description',
    'SIPRI TIV per unit': 'tiv_unit',
    'SIPRI TIV for total order': 'tiv_order',
    'SIPRI TIV of delivered weapons': 'tiv_delivered',
})

cols_to_drop = [c for c in df.columns if str(c).strip() in ['', '.1', '.2', '.3']]
print("Dropping:", cols_to_drop)
df = df.drop(columns=cols_to_drop)
print(df.columns.tolist())

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(df.shape)
df.head(5)

Dropping: [' ', ' .1', ' .2']
['Recipient', 'Supplier', 'year_order', 'num_ordered', 'weapon_designation', 'weapon_description', 'num_delivered', 'year_delivery', 'status', 'Comments', 'tiv_unit', 'tiv_order', 'tiv_delivered']
(29916, 13)


,recipient,supplier,year_order,num_ordered,weapon_designation,weapon_description,num_delivered,year_delivery,status,comments,tiv_unit,tiv_order,tiv_delivered
0,Afghanistan,Russia,2010,6.0,Mi-17,transport helicopter,6,2011,Second hand,Probably second-hand; Mi-17V1 version; bought ...,2.90,17.4,17.4
1,Afghanistan,Russia,2004,4.0,Mi-17,transport helicopter,4,2005,Second hand,Second-hand; aid; Mi-8MTV version,2.90,11.6,11.6
2,Afghanistan,Soviet Union,1960,50.0,BM-13 132mm,multiple rocket launcher,50,1960; 1961,Second hand,Second-hand,0.08,4.0,4.0
3,Afghanistan,Soviet Union,1978,500.0,T-55,tank,500,1979; 1980; 1981; 1982; 1983; 1984; 1985; 1986...,Second hand,Second-hand; aid,0.50,250.0,250.0
4,Afghanistan,Soviet Union,1987,40.0,MiG-21MF,fighter aircraft,40,1987; 1988; 1989; 1990,Second hand,Second-hand; probably incl some MiG-21UM,4.28,171.2,171.2


In [7]:
# doing some cleaning

# use order_tiv if available, but if not then delivered
# order => total order val, delivered => whats been delivered so far
df['tiv'] = df['tiv_order'].fillna(df['tiv_delivered'])

# drop vals with no TIV, recip, or supplier -> no use for a trade flow
df = df.dropna(subset=['tiv', 'recipient', 'supplier'])

# consistent names
df['recipient'] = df['recipient'].str.strip().str.title()
df['supplier'] = df['supplier'].str.strip().str.title()

In [8]:
# getting trade edges
edges = (
    df.groupby(['supplier', 'recipient', 'year_order'])
    .agg(
        tiv=('tiv', 'sum'),
        num_deals=('tiv', 'count'),
        weapon_types=('weapon_description', lambda x: list(x.dropna().unique()))
    )
    .reset_index()
    .rename(columns={'year_order': 'year'})
)

# makes years into ints
edges['year'] = pd.to_numeric(edges['year'], errors='coerce')
edges = edges.dropna(subset=['year'])
edges['year'] = edges['year'].astype(int)

# get top 10
top = (edges.groupby(['supplier','recipient'])
       .agg(total_tiv=('tiv','sum'), num_years=('year','nunique'))
       .reset_index()
       .sort_values('total_tiv', ascending=False)
       .head(10))
print(top)

           supplier       recipient  total_tiv  num_years
2886  United States           Japan   78462.74         72
2940  United States    Saudi Arabia   59005.28         60
1855         Russia           India   50905.91         37
2867  United States         Germany   50286.28         51
2950  United States     South Korea   49200.08         74
2883  United States          Israel   43671.13         67
2161   Soviet Union           India   42502.26         33
2960  United States          Taiwan   41109.57         73
2973  United States  United Kingdom   39647.18         66
2185   Soviet Union          Poland   38227.82         36


NOTE FROM THE ABOVE: Soviet Union appears, and so do non recognized states such as african states.

Default to post 1991, since soviet union and russia are different entities.

make toggle for recognized and non recognized orgs/states

In [9]:
# exporting to json

import numpy as np
import math

# so that json can read vals 
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating):
            if np.isnan(obj): return None
            return round(float(obj), 2)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if isinstance(obj, float) and math.isnan(obj): return None
        return super().default(obj)
    

edges_collapsed = (
    edges.groupby(['supplier', 'recipient'])
    .agg(
        total_tiv=('tiv', 'sum'),
        num_deals=('num_deals', 'sum'),
        years_active=('year', list),          # e.g. [2010, 2011, 2014, 2015]
        year_first=('year', 'min'),
        year_last=('year', 'max'),
        weapon_types=('weapon_types', lambda x: list(set(w for wl in x for w in wl)))
    )
    .reset_index()
)

# 1. Per-year edges (for filtering)
edges_out = edges[['supplier','recipient','year','tiv','num_deals','weapon_types']].copy()
edges_records = edges_out.to_dict(orient='records')

# 2. Collapsed edges (for default map view)
collapsed_records = edges_collapsed.to_dict(orient='records')

# 3. Node list with aggregate stats
nodes = pd.concat([
    edges_collapsed[['supplier']].rename(columns={'supplier':'country'}),
    edges_collapsed[['recipient']].rename(columns={'recipient':'country'})
]).drop_duplicates()

export_stats = (edges_collapsed.groupby('supplier')
    .agg(total_exported=('total_tiv','sum'))
    .reset_index().rename(columns={'supplier':'country'}))

import_stats = (edges_collapsed.groupby('recipient')
    .agg(total_imported=('total_tiv','sum'))
    .reset_index().rename(columns={'recipient':'country'}))

nodes = (nodes
    .merge(export_stats, on='country', how='left')
    .merge(import_stats, on='country', how='left')
    .fillna(0))

nodes_records = nodes.to_dict(orient='records')

# Export all three
with open('edges_by_year.json', 'w') as f:
    json.dump(edges_records, f, cls=NpEncoder)

with open('edges_collapsed.json', 'w') as f:
    json.dump(collapsed_records, f, cls=NpEncoder)

with open('nodes.json', 'w') as f:
    json.dump(nodes_records, f, cls=NpEncoder)

print(f"edges_by_year: {len(edges_records)} records")
print(f"edges_collapsed: {len(collapsed_records)} records")
print(f"nodes: {len(nodes_records)} countries")

edges_by_year: 16104 records
edges_collapsed: 3150 records
nodes: 258 countries


edges_per_year => trades per year

edges_collapsed => just all years together

In [10]:
import networkx as nx
import community.community_louvain as community_louvain

# Build directed graph from collapsed edges
G = nx.DiGraph()
for _, row in edges_collapsed.iterrows():
    G.add_edge(row['supplier'], row['recipient'], weight=row['total_tiv'])

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

# Centrality metrics
degree_cent = nx.degree_centrality(G)
in_degree = nx.in_degree_centrality(G)
out_degree = nx.out_degree_centrality(G)
betweenness = nx.betweenness_centrality(G, weight='weight')
pagerank = nx.pagerank(G, weight='weight')

# Louvain needs undirected graph
G_undirected = G.to_undirected()
partition = community_louvain.best_partition(G_undirected, weight='weight')

print(f"Clusters detected: {len(set(partition.values()))}")

Nodes: 258
Edges: 3150
Clusters detected: 4


Found 4 clusters, likely a Western bloc, a Russia/Soviet sphere, a China-led bloc, and a mixed/non-aligned group.

NOTE: the clusters are currently static. They are based on flattened edges, so the years don't matter. In the future, could implement different community detections for each decade.

this can be a good indicator for things like dependancy analysis. If a node is trading outside its cluster, this can signal unstable political dependance.

Below, we will fold all metrics into json

In [11]:
# Build enriched nodes dataframe
nodes['degree_centrality'] = nodes['country'].map(degree_cent).fillna(0)
nodes['in_degree'] = nodes['country'].map(in_degree).fillna(0)
nodes['out_degree'] = nodes['country'].map(out_degree).fillna(0)
nodes['betweenness'] = nodes['country'].map(betweenness).fillna(0)
nodes['pagerank'] = nodes['country'].map(pagerank).fillna(0)
nodes['cluster'] = nodes['country'].map(partition).fillna(-1).astype(int)

# Dependency index — for each importer, what % comes from top supplier
def dependency_index(country):
    imports = edges_collapsed[edges_collapsed['recipient'] == country]
    if imports.empty or imports['total_tiv'].sum() == 0:
        return 0, 'None'
    top = imports.loc[imports['total_tiv'].idxmax()]
    pct = top['total_tiv'] / imports['total_tiv'].sum()
    return round(pct, 3), top['supplier']

nodes[['dependency_index', 'top_supplier']] = nodes['country'].apply(
    lambda c: pd.Series(dependency_index(c))
)

with open('nodes.json', 'w') as f:
    json.dump(nodes.to_dict(orient='records'), f, cls=NpEncoder)


print(nodes.sort_values('betweenness', ascending=False).head(10))

                  country  total_exported  total_imported  degree_centrality  \
31                Czechia         2770.85         3435.13           0.260700   
126  United Arab Emirates         2350.41        41310.90           0.338521   
127        United Kingdom       151157.23        47411.97           0.607004   
114          Soviet Union       454223.79        30008.72           0.326848   
28                Croatia          109.46         1071.13           0.081712   
53                 Guyana            2.10          139.29           0.050584   
49                Germany       108618.19        73259.36           0.560311   
18               Bulgaria         1088.37        21562.68           0.214008   
62                 Israel        34229.54        52386.64           0.463035   
65                 Jordan         1413.13        12333.97           0.221790   

     in_degree  out_degree  betweenness  pagerank  cluster  dependency_index  \
31    0.066148    0.194553     0.099533

In [12]:
print("=== DATA SUMMARY ===")
print(f"Year range: {edges['year'].min()} – {edges['year'].max()}")
print(f"Unique suppliers: {edges['supplier'].nunique()}")
print(f"Unique recipients: {edges['recipient'].nunique()}")
print(f"Total TIV in dataset: {edges['tiv'].sum():,.0f}")
print(f"\n=== NODES ===")
print(f"Total nodes: {len(nodes)}")
print(f"Cluster distribution:\n{nodes['cluster'].value_counts()}")
print(f"\n=== TOP 5 EXPORTERS (all time) ===")
print(nodes.nlargest(5, 'total_exported')[['country','total_exported','cluster']])
print(f"\n=== TOP 5 IMPORTERS (all time) ===")
print(nodes.nlargest(5, 'total_imported')[['country','total_imported','cluster','dependency_index','top_supplier']])

=== DATA SUMMARY ===
Year range: 1940 – 2025
Unique suppliers: 138
Unique recipients: 255
Total TIV in dataset: 2,323,431

=== NODES ===
Total nodes: 258
Cluster distribution:
cluster
3    90
1    63
0    53
2    52
Name: count, dtype: int64

=== TOP 5 EXPORTERS (all time) ===
            country  total_exported  cluster
129   United States       865926.16        3
114    Soviet Union       454223.79        2
102          Russia       166778.57        1
46           France       162056.51        0
127  United Kingdom       151157.23        3

=== TOP 5 IMPORTERS (all time) ===
          country  total_imported  cluster  dependency_index   top_supplier
57          India       153004.11        1             0.333         Russia
103  Saudi Arabia        90451.88        0             0.652  United States
64          Japan        82070.79        3             0.956  United States
23          China        80105.61        1             0.466         Russia
38          Egypt        76542.99   

HERFINDAHL INDEX => Measures diversification, using this on imports, to see how much a country relies on others.

If not diversified => relies heavily on supplier

STATSCAN used it to measure export diversification: https://www150.statcan.gc.ca/n1/pub/13-605-x/2017001/article/54890-eng.htm

In [13]:
results = []
countries = edges['recipient'].unique()

for country in countries:
    country_imports = edges[edges['recipient'] == country]
    years = sorted(country_imports['year'].unique())
    
    for year in range(1955, 2024):
        window = country_imports[
            country_imports['year'].between(year-4, year)
        ]
        total = window['tiv'].sum()
        if total == 0:
            continue
        shares = window.groupby('supplier')['tiv'].sum() / total
        hhi = (shares**2).sum()
        top_supplier = shares.idxmax()
        top_share = shares.max()
        results.append({
            'country': country,
            'year': year,
            'hhi': round(hhi, 4),
            'top_supplier': top_supplier,
            'top_share': round(top_share, 3)
        })

hhi_df = pd.DataFrame(results)

# Using India as example
print(hhi_df[hhi_df['country']=='India'].tail(20))

     country  year     hhi top_supplier  top_share
1713   India  2004  0.5974       Russia      0.766
1714   India  2005  0.5468       Russia      0.729
1715   India  2006  0.3313       Russia      0.536
1716   India  2007  0.4710       Russia      0.670
1717   India  2008  0.4690       Russia      0.668
1718   India  2009  0.3633       Russia      0.535
1719   India  2010  0.3990       Russia      0.566
1720   India  2011  0.2991       Russia      0.453
1721   India  2012  0.2589       Russia      0.379
1722   India  2013  0.2482       Russia      0.377
1723   India  2014  0.3313       Russia      0.526
1724   India  2015  0.3109       Russia      0.475
1725   India  2016  0.3754       Russia      0.571
1726   India  2017  0.2207       France      0.310
1727   India  2018  0.2320       Russia      0.305
1728   India  2019  0.3067       Russia      0.471
1729   India  2020  0.2987       Russia      0.471
1730   India  2021  0.2895       Russia      0.479
1731   India  2022  0.4456     

In [14]:
# Saving to json
with open('hhi.json', 'w') as f:
    json.dump(hhi_df.to_dict(orient='records'), f, cls=NpEncoder)
print(f"HHI records: {len(hhi_df)}")

HHI records: 10020


Some Exploratory Data Analysis below:

running Louvain Community Detection on a per decade basis

for each decade, checking top 5 exporters per cluster

based on top 5, naming the cluster

In [15]:
decades = {
    '1940s': (1940, 1949),
    '1950s': (1950, 1959),
    '1960s': (1960, 1969),
    '1970s': (1970, 1979),
    '1980s': (1980, 1989),
    '1990s': (1990, 1999),
    '2000s': (2000, 2009),
    '2010s': (2010, 2019),
    '2020s': (2020, 2029),
}

for decade, (y1, y2) in decades.items():
    decade_edges = edges[edges['year'].between(y1, y2)]
    
    if len(decade_edges) < 10:
        print(f"{decade}: too sparse, skipping")
        continue
    
    G_dec = nx.Graph()
    for _, row in decade_edges.iterrows():
        if G_dec.has_edge(row['supplier'], row['recipient']):
            G_dec[row['supplier']][row['recipient']]['weight'] += row['tiv']
        else:
            G_dec.add_edge(row['supplier'], row['recipient'], weight=row['tiv'])
    
    partition_dec = community_louvain.best_partition(G_dec, weight='weight')
    n_clusters = len(set(partition_dec.values()))
    
    print(f"\n{decade}: {n_clusters} clusters, {G_dec.number_of_nodes()} countries")
    for c in range(n_clusters):
        members = [k for k,v in partition_dec.items() if v == c]
        # show top 5 by export volume so you can identify the cluster
        top = sorted(members, key=lambda x: nodes[nodes['country']==x]['total_exported'].values[0] if x in nodes['country'].values else 0, reverse=True)[:5]
        print(f"  cluster {c}: {top}")


1940s: 3 clusters, 54 countries
  cluster 0: ['United States', 'France', 'United Kingdom', 'Italy', 'Netherlands']
  cluster 1: ['Israel', 'Unknown Supplier(S)']
  cluster 2: ['Soviet Union', 'China', 'Czechoslovakia', 'Poland', 'North Korea']

1950s: 3 clusters, 104 countries
  cluster 0: ['Soviet Union', 'China', 'Czechoslovakia', 'Poland', 'North Korea']
  cluster 1: ['United States', 'France', 'Germany', 'Italy', 'Israel']
  cluster 2: ['United Kingdom', 'Netherlands', 'Switzerland', 'Sweden', 'Brazil']

1960s: 4 clusters, 139 countries
  cluster 0: ['Soviet Union', 'Czechoslovakia', 'Poland', 'Finland', 'North Korea']
  cluster 1: ['China', 'Pakistan', 'Albania', 'Tanzania']
  cluster 2: ['United States', 'France', 'United Kingdom', 'Germany', 'Italy']
  cluster 3: ['Unknown Supplier(S)', 'Cyprus', 'Gambia']

1970s: 3 clusters, 165 countries
  cluster 0: ['Soviet Union', 'Russia', 'Czechoslovakia', 'Poland', 'Finland']
  cluster 1: ['France', 'China', 'Spain', 'South Africa', 'Un

1940s => Soviet/Eastern Bloc (0), Western(1), Embargoed (2)

1950s => Secondary western (0), US-Led West (1), Soviet (2)

1960s => Soviet (0), West (1), Unidentified (2), Chinese Non-Aligned (3)

1970 => Soviet (0), West(1), Independent Arms Market (2)

1980 => Soviet (0), Non-Aligned Commercial (1), China (2), US-led West (3)

1990 => Western (0), Post-Soviet Surplus (1), Soviet(2), Gulf/French (3), Europe (4)

2000s => Bilateral Outliers (0), Franco-European (1), Anglosphere (2), Eurasian (3), Bilateral Outliers (4), German/Nordic (5)

2010s => Anglosphere + Allies (0), Eurasian Bloc (1), Continental EU (2), Secondary Exporters (3)

2020s => Chinese Bloc (0), Franco-Mediterranean (1), US-led West (2), Russian Isolation Cluster (3), Anglosphere Rump (4), Germanic/Nordic (5)

In [16]:
# actual code to store it

DECADE_LABELS = {
    '1940s': {0:'Soviet/Eastern Bloc', 1:'Western', 2:'Neutral/Embargoed'},
    '1950s': {0:'Secondary Western', 1:'US-led West', 2:'Soviet/Eastern Bloc'},
    '1960s': {0:'Soviet', 1:'US-led West', 2:'Fringe', 3:'Chinese Non-Aligned'},
    '1970s': {0:'Soviet', 1:'US-led West', 2:'Independent Arms Market'},
    '1980s': {0:'Soviet', 1:'Non-Aligned Commercial', 2:'Chinese Bloc', 3:'US-led West'},
    '1990s': {0:'US-led West', 1:'Post-Soviet Surplus', 2:'Soviet/Russian', 3:'Gulf-French', 4:'Continental Europe'},
    '2000s': {0:'Other', 1:'Franco-European', 2:'Anglosphere', 3:'Eurasian', 4:'Other', 5:'Germanic/Nordic'},
    '2010s': {0:'Anglosphere+Allies', 1:'Eurasian Bloc', 2:'Continental EU', 3:'Secondary Exporters'},
    '2020s': {0:'Chinese Bloc', 1:'Franco-Mediterranean', 2:'US-led West', 3:'Russian Isolation', 4:'Anglosphere Rump', 5:'Germanic/Nordic'},
}

decades = {
    '1940s': (1940, 1949),
    '1950s': (1950, 1959),
    '1960s': (1960, 1969),
    '1970s': (1970, 1979),
    '1980s': (1980, 1989),
    '1990s': (1990, 1999),
    '2000s': (2000, 2009),
    '2010s': (2010, 2019),
    '2020s': (2020, 2029),
}

# store results as dict keyed by country
decade_clusters = defaultdict(dict)

for decade, (y1, y2) in decades.items():
    decade_edges = edges[edges['year'].between(y1, y2)]
    
    if len(decade_edges) < 10:
        print(f"{decade}: too sparse, skipping")
        continue
    
    # build undirected weighted graph
    G_dec = nx.Graph()
    for _, row in decade_edges.iterrows():
        if G_dec.has_edge(row['supplier'], row['recipient']):
            G_dec[row['supplier']][row['recipient']]['weight'] += row['tiv']
        else:
            G_dec.add_edge(row['supplier'], row['recipient'], weight=row['tiv'])
    
    partition_dec = community_louvain.best_partition(G_dec, weight='weight')
    labels = DECADE_LABELS.get(decade, {})
    
    for country, cluster_id in partition_dec.items():
        label = labels.get(cluster_id, f'Other')
        decade_clusters[country][decade] = label

print(f"Countries with decade clusters: {len(decade_clusters)}")
print("\nSample — India:")
print(decade_clusters.get('India', {}))
print("\nSample — Ukraine:")
print(decade_clusters.get('Ukraine', {}))

Countries with decade clusters: 258

Sample — India:
{'1940s': 'Soviet/Eastern Bloc', '1950s': 'Secondary Western', '1960s': 'Soviet', '1970s': 'Soviet', '1980s': 'Soviet', '1990s': 'Soviet/Russian', '2000s': 'Eurasian', '2010s': 'Eurasian Bloc', '2020s': 'Chinese Bloc'}

Sample — Ukraine:
{'1950s': 'US-led West', '1980s': 'Soviet', '1990s': 'Soviet/Russian', '2000s': 'Eurasian', '2010s': 'Eurasian Bloc', '2020s': 'Germanic/Nordic'}


In [17]:
# Add decade clusters to nodes dataframe
for decade in decades.keys():
    nodes[f'cluster_{decade}'] = nodes['country'].map(
        lambda c: decade_clusters.get(c, {}).get(decade, 'No Data')
    )

# Re-export nodes with decade clusters included
with open('nodes.json', 'w') as f:
    json.dump(nodes.to_dict(orient='records'), f, cls=NpEncoder)

print("Saved. Sample columns:")
print(nodes.columns.tolist())
print("\nIndia:")
india = nodes[nodes['country']=='India'].iloc[0]
print({col: india[col] for col in nodes.columns if 'cluster' in col})

Saved. Sample columns:
['country', 'total_exported', 'total_imported', 'degree_centrality', 'in_degree', 'out_degree', 'betweenness', 'pagerank', 'cluster', 'dependency_index', 'top_supplier', 'cluster_1940s', 'cluster_1950s', 'cluster_1960s', 'cluster_1970s', 'cluster_1980s', 'cluster_1990s', 'cluster_2000s', 'cluster_2010s', 'cluster_2020s']

India:
{'cluster': np.int64(1), 'cluster_1940s': 'Soviet/Eastern Bloc', 'cluster_1950s': 'Secondary Western', 'cluster_1960s': 'Soviet', 'cluster_1970s': 'Soviet', 'cluster_1980s': 'Soviet', 'cluster_1990s': 'Soviet/Russian', 'cluster_2000s': 'Eurasian', 'cluster_2010s': 'Eurasian Bloc', 'cluster_2020s': 'Chinese Bloc'}


NOTE: the above indian timeline does tell the overall story BUT, this is only based on arms trade data. Even though India is currently in the "Chinese Bloc", they do have tensions, India simply depends on China for arms.

In [23]:
export_by_year = (edges.groupby(['supplier', 'year'])
    .agg(total_exported=('tiv','sum'))
    .reset_index().rename(columns={'supplier':'country'}))

import_by_year = (edges.groupby(['recipient', 'year'])
    .agg(total_imported=('tiv','sum'))
    .reset_index().rename(columns={'recipient':'country'}))

trade_balance = (export_by_year
    .merge(import_by_year, on=['country', 'year'], how='outer')
    .fillna(0))

trade_balance['balance'] = trade_balance['total_exported'] - trade_balance['total_imported']

print(trade_balance.head(10))

       country  year  total_exported  total_imported  balance
0  Afghanistan  1955             0.0          538.80  -538.80
1  Afghanistan  1956             0.0           30.08   -30.08
2  Afghanistan  1957             0.0            9.50    -9.50
3  Afghanistan  1958             0.0           97.80   -97.80
4  Afghanistan  1960             0.0           82.40   -82.40
5  Afghanistan  1961             0.0           87.00   -87.00
6  Afghanistan  1963             0.0           94.84   -94.84
7  Afghanistan  1964             0.0           76.50   -76.50
8  Afghanistan  1965             0.0          594.22  -594.22
9  Afghanistan  1967             0.0            7.60    -7.60


In [24]:
# EXPORTING TO JSON
with open('trade_balance.json', 'w') as f:
    json.dump(trade_balance.to_dict(orient='records'), f, cls=NpEncoder)

print(f"trade_balance records: {len(trade_balance)}")

trade_balance records: 8061


BELOW will be time series analysis on imports + cross referencing with conflict data

In [55]:
# Computing mean and std dev for each year + country
WINDOW = 8

# sort first so years are in order
trade_balance = trade_balance.sort_values(['country', 'year'])

# then for each country group, apply rolling stats
# .rolling(window=5, min_periods=3) gives 5-year window
# requiring at least 3 data points before computing
trade_balance['rolling_mean'] = (trade_balance
    .groupby('country')['total_imported']
    .transform(lambda x: x.rolling(window=WINDOW, min_periods=3).mean()))

# Standard Deviation
trade_balance['rolling_std'] = (trade_balance
        .groupby('country')['total_imported']
        .transform(lambda x: x.rolling(window=WINDOW, min_periods=3).std()))

# Z score (normalizing)
trade_balance['zscore'] = (
    (trade_balance['total_imported'] - trade_balance['rolling_mean']) 
    / trade_balance['rolling_std']
)

# By the Empirical Rule, 95 % of data should lie within 2 std devs
trade_balance['is_spike'] = trade_balance['zscore'] > 2


print(f"Spikes detected at z>2: {trade_balance['is_spike'].sum()}")
print(trade_balance[trade_balance['is_spike']]
      .sort_values('zscore', ascending=False)
      .head(15)[['country','year','total_imported','zscore']])

Spikes detected at z>2: 497
           country  year  total_imported    zscore
600        Belarus  2005          145.70  2.474874
5796        Russia  1995           72.00  2.474874
7337       Ukraine  2014           41.94  2.474874
6792   Switzerland  2021         2325.00  2.474801
1001      Bulgaria  2019          382.00  2.474729
6405  Soviet Union  1954         7200.00  2.474471
5052   North Yemen  1979         1335.00  2.474367
5876  Saudi Arabia  1965          514.35  2.474294
689        Belgium  2018         1898.42  2.474252
1660          Cuba  1960         1521.52  2.474113
3035       Hungary  2018          552.50  2.471949
5800        Russia  1999         1444.36  2.471824
1236        Canada  2022         4932.00  2.470488
5121        Norway  2008         2872.30  2.468759
3015       Hungary  1993          927.98  2.468633


As we can see, the algorithm found Cuba, 1960, which shows russian arms imports post Cuban Revolution.

Also, Ukraine 2014, after Russian Invasion.

Threshold validated against real events.

In [56]:
for threshold in [1.5, 2.0, 2.5, 3.0]:
    n = (trade_balance['zscore'] > threshold).sum()
    pct = n / trade_balance['zscore'].notna().sum() * 100
    print(f"z>{threshold}: {n} spikes ({pct:.1f}% of country-years)")

z>1.5: 939 spikes (12.6% of country-years)
z>2.0: 497 spikes (6.6% of country-years)
z>2.5: 0 spikes (0.0% of country-years)
z>3.0: 0 spikes (0.0% of country-years)


In [57]:
# Saving before moving on to cross referencing
with open('trade_balance.json', 'w') as f:
    json.dump(trade_balance.to_dict(orient='records'), f, cls=NpEncoder)

print(f"Records: {len(trade_balance)}")
print(f"Spikes: {trade_balance['is_spike'].sum()}")

Records: 8061
Spikes: 497


UCDP PART

In [60]:
ucdp = pd.read_csv('UCDP.csv')
print(ucdp.columns)

Index(['dyad_id', 'conflict_id', 'location', 'side_a', 'side_a_id',
       'side_a_2nd', 'side_b', 'side_b_id', 'side_b_2nd', 'incompatibility',
       'territory_name', 'year', 'intensity_level', 'type_of_conflict',
       'start_date', 'start_prec', 'start_date2', 'start_prec2', 'gwno_a',
       'gwno_a_2nd', 'gwno_b', 'gwno_b_2nd', 'gwno_loc', 'region', 'version'],
      dtype='str')


NOTE:

location => where conflict took place

side_a/side_b => two parties

year

In [61]:
print(ucdp[['location', 'side_a', 'side_b', 'year', 
            'intensity_level', 'type_of_conflict']].head(10))
print(f"Years: {ucdp['year'].min()} - {ucdp['year'].max()}")
print(f"Unique locations: {ucdp['location'].nunique()}")

           location                          side_a      side_b  year  \
0          Ethiopia          Government of Ethiopia        IGLF  1991   
1              Mali              Government of Mali        FPLA  1991   
2  DR Congo (Zaire)  Government of DR Congo (Zaire)       APCLS  2013   
3  DR Congo (Zaire)  Government of DR Congo (Zaire)       APCLS  2014   
4  DR Congo (Zaire)  Government of DR Congo (Zaire)       APCLS  2020   
5  DR Congo (Zaire)  Government of DR Congo (Zaire)       APCLS  2021   
6            Turkey            Government of Turkey         TAK  2016   
7             Kenya             Government of Kenya  Al-Shabaab  2015   
8             Kenya             Government of Kenya  Al-Shabaab  2016   
9             Kenya             Government of Kenya  Al-Shabaab  2017   

   intensity_level  type_of_conflict  
0                1                 3  
1                1                 3  
2                1                 4  
3                1                 4  
4

In [69]:
# Clean location — split multi-country locations into separate rows
ucdp_clean = ucdp.copy()

# split "India, Pakistan" type locations into separate rows
ucdp_clean = ucdp_clean.assign(
    location=ucdp_clean['location'].str.split(r',\s*')  # handles both ", " and ","
).explode('location')

ucdp_clean['location'] = ucdp_clean['location'].str.strip()

# extract side_b only when it's a government
ucdp_clean['state_b'] = ucdp_clean['side_b'].where(
    ucdp_clean['side_b'].str.startswith('Government of', na=False)
)

# clean side_a — remove "Government of " prefix to get country name
ucdp_clean['state_a'] = ucdp_clean['side_a'].str.replace('Government of ', '', regex=False).str.strip()
ucdp_clean['state_b_clean'] = ucdp_clean['state_b'].str.replace('Government of ', '', regex=False).str.strip()

print(ucdp_clean[['location', 'state_a', 'state_b_clean', 'year', 'intensity_level']].head(10))

           location           state_a state_b_clean  year  intensity_level
0          Ethiopia          Ethiopia           NaN  1991                1
1              Mali              Mali           NaN  1991                1
2  DR Congo (Zaire)  DR Congo (Zaire)           NaN  2013                1
3  DR Congo (Zaire)  DR Congo (Zaire)           NaN  2014                1
4  DR Congo (Zaire)  DR Congo (Zaire)           NaN  2020                1
5  DR Congo (Zaire)  DR Congo (Zaire)           NaN  2021                1
6            Turkey            Turkey           NaN  2016                1
7             Kenya             Kenya           NaN  2015                1
8             Kenya             Kenya           NaN  2016                1
9             Kenya             Kenya           NaN  2017                1


In [76]:
# collect all country-year conflict observations from all three sources
from_location = ucdp_clean[['location', 'year', 'intensity_level', 'type_of_conflict']].rename(columns={'location': 'country'})
from_location = from_location.assign(country=from_location['country'].str.split(r',\s*')).explode('country')

from_state_a = ucdp_clean[['state_a', 'year', 'intensity_level', 'type_of_conflict']].rename(columns={'state_a': 'country'})
from_state_a = from_state_a.assign(country=from_state_a['country'].str.split(r',\s*')).explode('country')

from_state_b = ucdp_clean[['state_b_clean', 'year', 'intensity_level', 'type_of_conflict']].dropna(subset=['state_b_clean']).rename(columns={'state_b_clean': 'country'})
from_state_b = from_state_b.assign(country=from_state_b['country'].str.split(r',\s*')).explode('country')

conflicts = pd.concat([from_location, from_state_a, from_state_b])
conflicts = conflicts.dropna(subset=['country'])
conflicts['country'] = conflicts['country'].str.strip()

# deduplicate — one row per country per year
conflicts = conflicts.sort_values('intensity_level', ascending=False)
conflicts = conflicts.drop_duplicates(subset=['country', 'year'], keep='first')

print(f"Conflict country-years: {len(conflicts)}")
print(conflicts.head(10))

Conflict country-years: 2122
                      country  year  intensity_level  type_of_conflict
3013                   Kuwait  1991                2                 2
3012                   Kuwait  1990                2                 2
2936                    Libya  1987                2                 2
2674  Vietnam (North Vietnam)  1979                2                 2
2399                     Iraq  1988                2                 2
2398                     Iraq  1987                2                 2
2397                     Iraq  1986                2                 2
2396                     Iraq  1985                2                 2
2395                     Iraq  1984                2                 2
2394                     Iraq  1983                2                 2


In [80]:
# manual mapping of UCDP names to SIPRI names
name_map = {
    'DR Congo (Zaire)': 'DR Congo',
    'Congo (Brazzaville)': 'Congo',
    'Myanmar (Burma)': 'Myanmar',
    'Bosnia-Herzegovina': 'Bosnia-Herzegovina',
    'United States of America': 'United States',
    'Russia (Soviet Union)': 'Russia',
    'Serbia (Yugoslavia)': 'Serbia',
    'Yugoslavia': 'Yugoslavia',
    'Rumania': 'Romania',
    'Iran (Persia)': 'Iran',
    'Turkey (Ottoman Empire)': 'Turkey',
    'Hyderabad': None,  # not a sovereign state in SIPRI
    'Vietnam (North)': 'Vietnam',
    'Vietnam (South)': 'Vietnam',
    'Yemen (North Yemen)': 'Yemen',
    'Yemen (South Yemen)': 'Yemen',
    'Cameroon (Cameroun)': 'Cameroon',
    'Zimbabwe (Rhodesia)': 'Zimbabwe',
    'Sri Lanka (Ceylon)': 'Sri Lanka',
    'Cambodia (Kampuchea)': 'Cambodia',
    'Burkina Faso (Upper Volta)': 'Burkina Faso',
    'Madagascar (Malagasy)': 'Madagascar',
    'Benin (Dahomey)': 'Benin',
    'Thailand (Siam)': 'Thailand',
    "Cote D'Ivoire" : 'Ivory Coast',
    'Ivory Coast' : 'Ivory Coast',
    'DR Congo': 'Dr Congo',
    'Trinidad and Tobago': 'Trinidad And Tobago',
    'Vietnam (North Vietnam)': 'South Vietnam',
    'Turkey': 'Turkiye',
}

conflicts['country'] = conflicts['country'].replace(name_map)
conflicts = conflicts.dropna(subset=['country'])

# check overlap now
trade_countries = set(trade_balance['country'].unique())
conflict_countries = set(conflicts['country'].unique())
matched = trade_countries & conflict_countries
unmatched = conflict_countries - trade_countries

print(f"Matched: {len(matched)}")
print(f"Still unmatched: {len(unmatched)}")
print(sorted(unmatched)[:20])

Matched: 121
Still unmatched: 1
['Ivory Coast']


In [88]:
print(conflicts[conflicts['country'] == 'Ivory Coast'])

          country  year  intensity_level  type_of_conflict
17    Ivory Coast  2011                1                 3
3325  Ivory Coast  2004                1                 3
3324  Ivory Coast  2003                1                 3
3322  Ivory Coast  2002                1                 3


Still 4 ivory coast ones, but not too much so wont affect anything

BELOW, I am finding spikes

In [102]:
spikes = trade_balance[trade_balance['is_spike']].copy()


# has_conflict_nearby(country, year, window=2) checks a countries conflicts withing [year-window, year+window]
# default window=2
def has_conflict_nearby(country, year, window=3):
    country_conflicts = conflicts[conflicts['country'] == country]
    if country_conflicts.empty:
        return False, None
    nearby = country_conflicts[
        country_conflicts['year'].between(year, year + window)
    ]
    if nearby.empty:
        return False, None
    closest = nearby.iloc[(nearby['year'] - year).abs().argsort().iloc[0]]
    return True, int(closest['year'])

spikes[['conflict_nearby', 'conflict_year']] = spikes.apply(
    lambda row: pd.Series(has_conflict_nearby(row['country'], row['year'])),
    axis=1
)

# RESULTS
total_spikes = len(spikes)
conflict_spikes = spikes['conflict_nearby'].sum()
pct = conflict_spikes / total_spikes * 100

print(f"\n=== SPIKE-CONFLICT CORRELATION ===")
print(f"Total spikes: {total_spikes}")
print(f"Spikes near conflict: {conflict_spikes} ({pct:.1f}%)")
print(f"\nTop validated spikes:")
print(spikes[spikes['conflict_nearby']]
      .sort_values('zscore', ascending=False)
      .head(15)[['country', 'year', 'total_imported', 'zscore', 'conflict_year']])

# FINDING BASELINE TO COMPARE
print("\n")


all_things = trade_balance[['country', 'year']].copy()
all_things[['conflict_nearby', 'conflict_year']] = all_things.apply(
    lambda row: pd.Series(has_conflict_nearby(row['country'], row['year'])),
    axis=1
)
baseline = all_things['conflict_nearby'].mean() * 100
print(baseline)


=== SPIKE-CONFLICT CORRELATION ===
Total spikes: 497
Spikes near conflict: 135 (27.2%)

Top validated spikes:
          country  year  total_imported    zscore  conflict_year
7337      Ukraine  2014           41.94  2.474874         2014.0
5796       Russia  1995           72.00  2.474874         1995.0
1660         Cuba  1960         1521.52  2.474113         1961.0
5800       Russia  1999         1444.36  2.471824         1999.0
7345      Ukraine  2022         7867.17  2.468432         2022.0
2311     Ethiopia  1977         2371.98  2.468410         1977.0
5151         Oman  1974          607.58  2.466130         1974.0
4860    Nicaragua  1981          230.13  2.464990         1982.0
3215         Iran  1964         2009.14  2.461147         1966.0
6581        Sudan  1968          576.56  2.460914         1968.0
140       Algeria  2006         4645.00  2.458830         2006.0
1149     Cameroon  2012          154.82  2.458759         2015.0
3868         Laos  1961           70.18  2.4

Baseline being similar to spike-conflict resolution would imply that the spikes do not significantly relate to conflicts. This was tested with ranges of [year, year + 3], [year - 3, year], [year - 1, year + 1], [year - 2, year + 2], [year - 3, year + 3] and yielded similar results.

Built an anomaly detection system across 8,061 country-years of arms-import data, identifying 497 statistically significant procurement surges. Tested the hypothesis that import spikes correlate with armed conflict and found little evidence of predictive power, motivating exploration of network- and supplier-based indicators.

arms import anomalies occur across diverse geopolitical contexts, with conflict being one but not the dominant driver

Below:

Built a retrieval-augmented AI system that enriches detected arms-trade anomalies with automatically generated geopolitical context.

In [114]:
# using AI to give each spike historical background

from google import genai

import os
from dotenv import load_dotenv
load_dotenv()
API = os.getenv("API")

client = genai.Client(api_key=API)

response = client.models.generate_content(
    model="gemma-4-26b-a4b-it",
    contents=f"""
    Country: Ethiopia
    Year: 1977
    Import value: 2371.98
    Z-score: 2.468410

    Explain possible geopolitical events that may have contributed
    to this unusual increase in military imports.
    Keep under 50 words.
    """
)

spikes1 = spikes.head(10).copy()

spikes1['explanation'] = spikes1.apply(
    lambda row: client.models.generate_content(
        model="gemma-4-26b-a4b-it",
        contents=f"""
        Country: {row['country']}
        Year: {row['year']}
        Import value: {row['total_imported']}
        Z-score: {row['zscore']}

        Nearby conflict year: {row['conflict_year']}

        Explain possible geopolitical events that may have contributed
        to this unusual increase in military imports.
        Keep under 50 words.
        """
    ).text,
    axis=1
)

print(spikes1.head(10))

             country  year  total_exported  total_imported  balance  \
8        Afghanistan  1965            0.00          594.22  -594.22   
17       Afghanistan  1978            0.00         1094.30 -1094.30   
26       Afghanistan  1988            0.00         3791.30 -3791.30   
54   African Union**  2021            0.66            8.40    -7.74   
69           Albania  1962            0.00          398.90  -398.90   
72           Albania  1965            0.00          666.90  -666.90   
110          Algeria  1976            0.00         1138.06 -1138.06   
111          Algeria  1977            0.00         2097.89 -2097.89   
112          Algeria  1978            0.00         3536.09 -3536.09   
140          Algeria  2006            0.00         4645.00 -4645.00   

     rolling_mean  rolling_std    zscore  is_spike  conflict_nearby  \
8       134.04250   188.650776  2.439309      True            False   
17      267.43750   345.677959  2.392002      True             True   
26   

In [119]:
for spike in spikes1['explanation']:
    print(spike)

The spike likely reflects Cold War competition. Both the United States and the Soviet Union provided military aid and hardware to Afghanistan to expand their regional influence. This strategic maneuvering aimed to secure the country as a vital buffer state amidst global superpower tensions.
The 1978 Saur Revolution, a communist coup by the PDPA, shifted Afghanistan toward the Soviet bloc. This political upheaval, alongside rising internal instability and the need to consolidate the new regime's power, likely triggered the significant surge in military imports.
The surge likely reflects the intensifying Soviet-Afghan War. Following the 1988 Geneva Accords, the Soviet-backed government likely increased military imports to bolster defenses and maintain regime stability against Mujahideen insurgents as the Soviet Union prepared for its scheduled withdrawal.
The spike likely reflects rising regional instability, including the Tigray conflict in Ethiopia and escalating insurgencies in the Sa

Above was a test, to see how good the responses are.

Problem is, it takes a long time to run. I am going to try to async it

In [121]:
import asyncio
import time

async def get_explanation(client, row):
    try:
        response = await client.aio.models.generate_content(
            model="gemma-4-26b-a4b-it",
            contents=f"""Country: {row['country']}
Year: {row['year']}
Import value: {row['total_imported']:.0f}
Z-score: {row['zscore']:.2f}
Nearby conflict year: {row['conflict_year']}

Explain possible geopolitical events that may have contributed to this unusual increase in military imports. Keep under 50 words."""
        )
        return response.text
    except Exception as e:
        print(f"Error for {row['country']} {row['year']}: {e}")
        return None

async def run_all(spikes_df, batch_size=14):
    results = []
    rows = [row for _, row in spikes_df.iterrows()]
    
    # process in batches to avoid rate limits
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(rows)-1)//batch_size + 1}...")
        tasks = [get_explanation(client, row) for row in batch]
        batch_results = await asyncio.gather(*tasks)
        results.extend(batch_results)
        # small delay between batches for rate limiting
        if i + batch_size < len(rows):
            await asyncio.sleep(60)
    
    return results

# run it
start = time.time()
explanations = await run_all(spikes)
spikes['explanation'] = explanations
print(f"Done in {time.time()-start:.1f}s")
print(f"Failed: {spikes['explanation'].isna().sum()}")

Processing batch 1/36...
Processing batch 2/36...
Processing batch 3/36...
Processing batch 4/36...
Processing batch 5/36...
Processing batch 6/36...
Processing batch 7/36...
Processing batch 8/36...
Processing batch 9/36...
Processing batch 10/36...
Processing batch 11/36...
Processing batch 12/36...
Processing batch 13/36...
Processing batch 14/36...
Processing batch 15/36...
Processing batch 16/36...
Processing batch 17/36...
Processing batch 18/36...
Processing batch 19/36...
Processing batch 20/36...
Processing batch 21/36...
Processing batch 22/36...
Processing batch 23/36...
Processing batch 24/36...
Processing batch 25/36...
Processing batch 26/36...
Processing batch 27/36...
Processing batch 28/36...
Processing batch 29/36...
Processing batch 30/36...
Processing batch 31/36...
Processing batch 32/36...
Processing batch 33/36...
Processing batch 34/36...
Processing batch 35/36...
Processing batch 36/36...
Done in 3489.6s
Failed: 0


No failed, gonna print the first 5

In [132]:
count = 0
for exp, country, year, zscore in spikes[['explanation', 'country', 'year', 'zscore']].sort_values(by='zscore', ascending=False).itertuples(index=False, name=None):
    print(country, year, exp)
    count += 1
    if count > 5:
        break

Belarus 2005 The 2005 spike likely reflects heightened security concerns driven by the "Color Revolutions" in neighboring post-Soviet states, notably Ukraine's Orange Revolution. Fearing regional instability and potential regime change, Belarus likely increased military imports to bolster domestic defense and strengthen strategic ties with Russia.
Russia 1995 The 1995 spike likely reflects Russia's involvement in the First Chechen War (1994–1996). The ongoing conflict necessitated urgent procurement of military hardware, equipment, and supplies to bolster combat capabilities and replenish stocks during intensive operations against separatist forces.
Ukraine 2014 The 2014 surge resulted from Russia's annexation of Crimea and the outbreak of war in the Donbas region. These geopolitical crises, following the Euromaidan Revolution, necessitated a rapid increase in military imports to bolster Ukraine's national defense against escalating territorial aggression.
Switzerland 2021 The increase

In [ ]:
# Exporting

with open('spikes.json', 'w') as f:
    json.dump(spikes.to_dict(orient='records'), f, cls=NpEncoder)

print(f"Spikes exported: {len(spikes)}")
print(spikes.columns.tolist())

Spikes exported: 497
['country', 'year', 'total_exported', 'total_imported', 'balance', 'rolling_mean', 'rolling_std', 'zscore', 'is_spike', 'conflict_nearby', 'conflict_year', 'explanation']


In [137]:
#fixing some problem with trade_balance

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating):
            if np.isnan(obj): return None
            return round(float(obj), 2)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if isinstance(obj, float) and math.isnan(obj): return None
        return super().default(obj)

import math

# replace NaN with None which serializes to null in JSON
trade_balance['rolling_mean'] = trade_balance['rolling_mean'].where(
    trade_balance['rolling_mean'].notna(), None
)
trade_balance['rolling_std'] = trade_balance['rolling_std'].where(
    trade_balance['rolling_std'].notna(), None
)
trade_balance['zscore'] = trade_balance['zscore'].where(
    trade_balance['zscore'].notna(), None
)


# easier approach — just use pandas fillna
trade_balance = trade_balance.where(pd.notnull(trade_balance), None)

import math

def clean_nan(obj):
    if isinstance(obj, float) and math.isnan(obj):
        return None
    return obj

records = trade_balance.to_dict(orient='records')
cleaned = [{k: clean_nan(v) for k, v in row.items()} for row in records]

with open('trade_balance.json', 'w') as f:
    json.dump(cleaned, f, cls=NpEncoder)

print("done")

done


In [139]:
# same prob with spikes
spikes = spikes.where(pd.notnull(spikes), None)

records = spikes.to_dict(orient='records')
cleaned = [{k: clean_nan(v) for k, v in row.items()} for row in records]

with open('spikes.json', 'w') as f:
    json.dump(cleaned, f, cls=NpEncoder)

print("done")

done
